# NNDM leave-one-out spatial validation lab

This lab constructs one test fold per training observation and greedily removes additional training points so held-out-to-training nearest distances better reflect distances encountered across a target prediction domain.

In [ ]:
from pathlib import Pathimport sys_root = Path.cwd()if not (_root / 'implementations').exists():    _root = _root.parentsys.path.insert(0, str(_root / 'implementations' / 'python' / 'nndm'))import numpy as np
import matplotlib.pyplot as plt
from nndm import nndm_loo

## Clustered training sample and target prediction domain

In [ ]:
rng=np.random.default_rng(5)
centres=np.array([[0.,0.],[100.,0.],[0.,100.],[100.,100.]])
training=np.vstack([c+rng.normal(0,2,size=(10,2)) for c in centres])
prediction=rng.uniform(-10,110,size=(800,2))
plt.figure(figsize=(7,6)); plt.scatter(prediction[:,0],prediction[:,1],s=3,alpha=.15,label='prediction domain'); plt.scatter(training[:,0],training[:,1],s=18,label='training'); plt.legend(); plt.axis('equal'); plt.show()

## Run NNDM

Choose phi from a defensible estimate of spatial autocorrelation range. The value below is illustrative only.

In [ ]:
phi=30.0
result=nndm_loo(training,prediction,autocorrelation_range=phi,min_train=.5)
print('additional exclusions:',result['n_removed'])
print('mean retained training size:',np.mean([len(f['train']) for f in result['folds']]))
print('minimum retained training size:',min(len(f['train']) for f in result['folds']))

## Compare nearest-distance distributions

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(result['prediction_nearest_distances'],bins=35,density=True,alpha=.5,label='prediction to training')
plt.hist(result['cv_nearest_distances'],bins=35,density=True,alpha=.5,label='held-out to retained training')
plt.xlabel('Nearest-neighbour distance (coordinate units)'); plt.ylabel('Density'); plt.legend(); plt.show()

## Inspect fold exclusions

Each fold holds out one observation. Additional excluded observations are specific to that fold; they remain available in other folds.

In [ ]:
fold_id=0
fold=result['folds'][fold_id]
plt.figure(figsize=(7,6))
plt.scatter(training[:,0],training[:,1],s=12,c='lightgray',label='all training')
plt.scatter(training[fold['train'],0],training[fold['train'],1],s=18,label='retained analysis')
plt.scatter(training[fold['excluded'],0],training[fold['excluded'],1],s=35,marker='x',label='additional exclusions')
plt.scatter(training[fold['test'],0],training[fold['test'],1],s=90,marker='*',label='held out')
plt.legend(); plt.axis('equal'); plt.show()

## Important limitations

This is a projected Euclidean teaching implementation. It does not infer phi, sample target polygons, support geodesic distances or scale to large datasets. A minimum-training fraction can stop matching before the target ECDF is reached. This validation design does not correct covariate shift, sampling bias, temporal dependence or model leakage. It is not a substitute for independent probability-sample map validation.